# Indic Document Intelligence with Kimi K3 on Amazon Bedrock

This notebook shows how to use **Kimi K3** on **Amazon Bedrock** to read scanned Hindi documents and turn them into structured data, using the **India cross-Region inference profile** (`in.moonshotai.kimi-k3`) so that inference stays within India (Mumbai `ap-south-1` and Hyderabad `ap-south-2`).

You will:

1. Call Kimi K3 directly with the OpenAI Python SDK on the `bedrock-runtime` endpoint's [Chat Completions API](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-chat-completions.html), as the [Kimi K3 model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-moonshot-ai-kimi-k3.html) recommends, to see what the model gives you on its own for a scanned Hindi electricity bill.
2. Define a **schema** for the bill and use the sample's small extraction pipeline to get back a full transcription, structured fields as JSON, and an English summary.
3. Check the extracted fields against hand-labelled ground truth.
4. Ask follow-up questions about the same document in **Hinglish** (code-mixed Hindi-English).
5. Repeat the flow on a hand-filled Hindi KYC form with a second schema, and on any document of your own with no schema at all.

> **Note:** All documents in `sample_data/` are **synthetic**. Every organisation, person, address and number is fictional, and identity numbers are masked. Do not use real personal documents with this sample unless you have the right to process them.

## Prerequisites

First, follow the repository-level [prerequisite setup instructions](../../../README.md) - which also discuss how to configure your **Python notebook kernel and installed libraries**.

For this sample in particular, you'll need:
- Access to `ap-south-1` (Mumbai) Region in your AWS Account
- [IAM permissions](https://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html) to `bedrock:InvokeModel` on the `in.moonshotai.kimi-k3` inference profile in your target (`ap-south-1`) region, and the underlying foundation model in **both** India Regions, plus `bedrock:CallWithBearerToken`.

For more information and access troubleshooting, see [Request access to models](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html) in the Amazon Bedrock User Guide.

## Setup

To get started, run the below code cell to load the libraries we'll use, sample code modules, and initial settings (model ID, Region, reasoning effort):

We'll enable Jupyter's [autoreload](https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html) extension, which reloads module code between cells so you can edit local `.py` files without needing to restart the notebook kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

# Python Built-Ins:
import json
from dataclasses import asdict
from pathlib import Path

# External Dependencies:
from aws_bedrock_token_generator import provide_token

# Local Utilities:
from indic_doc_intel import IndicDocumentIntelligence, Settings, load_image
from indic_doc_intel.evaluation import compare_fields, load_expected
from openai import OpenAI

# Local folder with the synthetic sample documents and their ground-truth labels:
SAMPLE_DATA = Path.cwd() / "sample_data"

# Defaults: model in.moonshotai.kimi-k3, Region ap-south-1, model-default reasoning effort (max).
# Override with KIMI_MODEL_ID, BEDROCK_REGION and KIMI_REASONING_EFFORT, or pass
# Settings(reasoning_effort="low") to trade a little quality for fewer output tokens.
settings = Settings.from_env()
print("\nLoaded settings:")
print(json.dumps(asdict(settings), indent=2))

Defines two small display helpers used in later cells: `show_json` prints results, and `show_comparison` prints a field-by-field ground-truth check.

In [ ]:
def show_json(data):
    """Pretty-print a dict, keeping Hindi text readable (ensure_ascii=False)."""
    print(json.dumps(data, ensure_ascii=False, indent=2))


def show_comparison(rows):
    """Print one line per field: OK if the extracted value matches ground truth, DIFF otherwise."""
    width = max(len(row["field"]) for row in rows)
    for row in rows:
        mark = "OK  " if row["match"] else "DIFF"
        print(
            f"{mark}  {row['field']:<{width}}  expected={row['expected']!r:<30} extracted={row['extracted']!r}"
        )
    # Summary line, e.g. "8/8 fields match ground truth"
    matched = sum(row["match"] for row in rows)
    print(f"\n{matched}/{len(rows)} fields match ground truth")

## Step 1: Look at the sample document

A synthetic electricity bill (बिजली बिल) from a fictional distribution company in Jaipur. Note that the dates and the units consumed use **Devanagari digits** (e.g. `०५/१०/२०२६`, `२८८`), while other numbers use Western digits, as is common on real Indian documents.

<img src="sample_data/electricity_bill_hi.png" width="520" alt="Synthetic Hindi electricity bill">

## Step 2: Call Kimi K3 directly

Before adding any structure, here is a plain Kimi K3 call. Amazon Bedrock exposes an OpenAI-compatible Chat Completions endpoint on `bedrock-runtime`, so the standard OpenAI Python SDK works once you point `base_url` at it. `provide_token()` turns your AWS credentials into a short-term Amazon Bedrock API key locally, so no long-term key is needed.

The only helper used here is `load_image()`, which downsizes the scan if needed and base64-encodes it as an image content part.

In [ ]:
bill_image = SAMPLE_DATA / "electricity_bill_hi.png"

# The standard OpenAI SDK, pointed at the OpenAI-compatible endpoint of bedrock-runtime.
client = OpenAI(
    base_url=f"https://bedrock-runtime.{settings.region}.amazonaws.com/openai/v1",
    api_key=provide_token(region=settings.region),
)

response = client.chat.completions.create(
    model=settings.model_id,
    messages=[
        {
            "role": "user",
            # The model card recommends placing images before text.
            "content": [
                load_image(bill_image),
                {
                    "type": "text",
                    "text": "Read this document and extract its key fields as JSON.",
                },
            ],
        }
    ],
    # Counts reasoning tokens as well as the answer, so leave generous headroom.
    max_completion_tokens=settings.max_tokens,
)
print(response.choices[0].message.content)

Kimi K3 reads the Hindi bill well on its own. For an application that consumes the output, though, a bare call leaves several gaps:

- **Field names are up to the model**, so they can change between runs and documents, and downstream code can't rely on them.
- **Formats aren't normalised.** Dates, Devanagari digits and amounts come back however the model chooses.
- **The JSON may be wrapped** in markdown fences or prose, and nothing retries if it doesn't parse.
- **No transcription, summary or uncertainty flags** unless you ask for them every time.
- **Nothing to compare against ground truth**, so prompt or model changes are hard to evaluate.

The rest of this notebook adds a small prompt pipeline over the same call ([`indic_doc_intel/pipeline.py`](indic_doc_intel/pipeline.py)). It adds a system prompt with extraction rules, builds the prompt from a schema you define, parses the JSON (asking the model to repair it once if needed), fills in missing fields as `null`, keeps the conversation for follow-up questions, adds up token usage and refreshes the API key before each request.

## Step 3: Define a schema for the document

A schema is how you tell the pipeline what to extract. It is a plain dict with a `description` of the document type and a `fields` dict that maps each output key to a short instruction, which is passed to the model verbatim. Writing a schema is the main thing you do to support a new document type, so it lives here in the notebook rather than in the library.

In [ ]:
ELECTRICITY_BILL_SCHEMA = {
    "description": "An Indian electricity (bijli) bill issued by a state power distribution company.",
    "fields": {
        "distribution_company": "Name of the electricity board or distribution company, in English.",
        "consumer_name": "Account holder name exactly as printed, in the original script.",
        "consumer_name_english": "Account holder name transliterated into English (Latin script).",
        "consumer_number": "Consumer / account number, digits only.",
        "address": "Supply address translated into English.",
        "billing_period": "Billing period as 'YYYY-MM-DD to YYYY-MM-DD'.",
        "bill_date": "Bill issue date as YYYY-MM-DD.",
        "due_date": "Payment due date as YYYY-MM-DD.",
        "units_consumed_kwh": "Units consumed in this period, as a number.",
        "amount_due_inr": "Total payable amount on or before the due date, as a number.",
        "amount_after_due_date_inr": "Amount payable after the due date, as a number.",
    },
}

## Step 4: Extract structured fields with the pipeline

`pipeline.extract()` makes the same kind of Chat Completions call as Step 2, but builds the prompt from `ELECTRICITY_BILL_SCHEMA` and adds the system prompt's extraction rules. Kimi K3 returns one JSON object containing the transcription, exactly the requested fields (dates normalised to ISO format, Devanagari digits converted, names transliterated) and an English summary.

Creates the pipeline, sends the scanned Hindi bill to Kimi K3 with the schema, and prints the structured fields it extracted.

In [ ]:
# Wraps a Kimi K3 client for the bedrock-runtime endpoint, built from the same settings.
pipeline = IndicDocumentIntelligence(settings)

# One call to Kimi K3: sends the image plus an extraction prompt built from the schema,
# and parses the JSON reply into a DocumentResult.
bill = pipeline.extract(bill_image, schema=ELECTRICITY_BILL_SCHEMA)

# Structured fields: ISO dates, Western digits, names transliterated into English.
show_json(bill.fields)

Shows the rest of the extraction result: the detected languages, the English summary and the start of the Hindi transcription.

In [ ]:
# The same reply also contains the detected languages, an English summary
# and a full transcription of the document in its original script.
print("Detected languages:", bill.data["detected_languages"])
print("\nEnglish summary:\n", bill.english_summary)
print("\nTranscription (first 600 characters):\n", bill.data["transcription"][:600])

## Step 5: Check against ground truth

`sample_data/expected/` holds hand-labelled values for each sample. Comparing against them is a simple way to catch regressions when you change prompts or models.

Scores the extraction against the hand-labelled ground truth for the bill.

In [ ]:
# Compare the extracted fields with the hand-labelled values in
# sample_data/expected/electricity_bill_hi.json (numbers and spacing are normalised).
show_comparison(compare_fields(bill.fields, load_expected(bill_image)))

## Step 6: Ask follow-up questions in Hinglish

Many users in India type in Hinglish: Hindi words written in Latin script, mixed with English. `pipeline.ask()` continues the same conversation, so Kimi K3 still has the original image in context, and is asked to reply in the user's own language style.

Asks three Hinglish questions about the bill, with one Kimi K3 call per question, and prints each answer.

In [ ]:
# Follow-up questions in Hinglish (Hindi written in Latin script, mixed with English).
questions = [
    "Bill kitne ka hai aur last date kab hai? Late hua toh kitna dena padega?",
    "Is mahine kitne units use hue aur meter number kya hai?",
    "Agar main bill pay nahi karun toh kya hoga?",
]

# Each ask() continues the extraction conversation, so the model still sees the bill image,
# and replies in the same style as the question (Hinglish in, Hinglish out).
for question in questions:
    print(f"Q: {question}\nA: {pipeline.ask(bill, question)}\n")

## Step 7: A hand-filled KYC form

The second sample is a synthetic address-update / KYC form for a fictional cooperative bank, filled in by hand in blue ink, with tick boxes, a masked Aadhaar number and a signature. The same pipeline is used with a second schema for this form.

<img src="sample_data/kyc_form_hi.png" width="520" alt="Synthetic Hindi KYC form">

Defines the schema for the KYC form. Only the document description and field instructions change.

In [ ]:
KYC_FORM_SCHEMA = {
    "description": "An Indian address-update / KYC application form filled in Hindi.",
    "fields": {
        "applicant_name": "Applicant name exactly as written, in the original script.",
        "applicant_name_english": "Applicant name transliterated into English (Latin script).",
        "father_or_spouse_name_english": "Father's or spouse's name transliterated into English.",
        "date_of_birth": "Date of birth as YYYY-MM-DD.",
        "gender": "One of 'male', 'female', 'other'.",
        "mobile_number": "10-digit mobile number, digits only.",
        "address": "Full address translated into English.",
        "pin_code": "6-digit PIN code.",
        "id_document_type": "Type of identity document referenced, in English.",
        "id_document_number": "Identity document number exactly as written.",
        "declaration_signed": "true if the declaration is signed, otherwise false.",
    },
}

Extracts the hand-filled KYC form with `KYC_FORM_SCHEMA`, prints the fields, and checks them against ground truth.

In [ ]:
kyc_image = SAMPLE_DATA / "kyc_form_hi.png"

# Same pipeline, different schema.
kyc = pipeline.extract(kyc_image, schema=KYC_FORM_SCHEMA)

show_json(kyc.fields)
print()
# Score against sample_data/expected/kyc_form_hi.json
show_comparison(compare_fields(kyc.fields, load_expected(kyc_image)))

Asks one Hinglish question about the KYC form and prints the answer.

In [ ]:
# A Hinglish follow-up question about the KYC form (continues the KYC conversation).
print(
    pipeline.ask(
        kyc, "Is form mein naya address kya likha hai aur kaunsa ID proof diya hai?"
    )
)

## Step 8: Bring your own document (schema-free mode)

Without a schema, Kimi K3 chooses the most important key-value pairs itself. This is useful for exploring a new document type before you write a schema for it. Point `my_image` at your own (non-sensitive) scanned document to try it.

Runs schema-free extraction. Point `my_image` at your own non-sensitive scan to try a new document type.

In [ ]:
my_image = SAMPLE_DATA / "electricity_bill_hi.png"  # replace with your own image path

# No schema: Kimi K3 picks the document type and the key fields itself.
generic = pipeline.extract(my_image)
print("Document type:", generic.data["document_type"])
show_json(generic.fields)

## Token usage

Each `DocumentResult` adds up the token usage reported by the Chat Completions API across extraction and follow-up questions. `completion_tokens` includes `reasoning_tokens`, and `prompt_tokens` includes `cached_prompt_tokens`, which are billed at the lower cache-read rate. Multiply by the current Kimi K3 rates on the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/) to estimate cost.

Prints the token usage of each document so you can estimate cost.

In [ ]:
# Each result adds up the tokens used by its extraction call and any follow-up questions.
for name, result in [
    ("electricity_bill", bill),
    ("kyc_form", kyc),
    ("generic", generic),
]:
    print(f"{name:<17} {result.usage}")

## Clean up

This notebook does not create any AWS resources. You are charged only for the Amazon Bedrock tokens consumed by the calls above.

## Next steps

- Define a schema dict for your own document type, as in Steps 3 and 7, plus a ground-truth file in `sample_data/expected/`.
- Try documents in other Indic languages (Marathi, Tamil, Bengali and others) in schema-free mode.
- For production workloads, add human review for low-confidence fields (see `uncertain_fields` in each result) and store results in a system such as Amazon DynamoDB or Amazon S3.